# ESP32 Distributed Cluster - Comprehensive Test Suite

This notebook runs all 14 comprehensive tests for the ESP32 cluster system:
- Basic task execution
- Canvas primitives (GROUP, CHAIN, CHORD)
- Dual-core execution
- Peripheral control (I2C, SPI, UART, CAN)
- GPIO & ADC
- System monitoring
- **Dynamic multiline Python code upload**
- Error handling

**IMPORTANT:** This notebook requires **Anaconda Python** kernel with working `pyserial`. 
The MS Store Python kernel has broken serial module support.

Run cells sequentially to test all features.

In [3]:
# Setup: Import dependencies and locate broccoli_cluster
import sys
import os
import time

print(f"Python: {sys.executable}")

# Check if pyserial is working
try:
    import serial
    has_serial_class = hasattr(serial, 'Serial')
    has_serial_exc = hasattr(serial, 'SerialException')
    print(f"pyserial installed: Yes")
    print(f"  - Serial class: {has_serial_class}")
    print(f"  - SerialException: {has_serial_exc}")
    if not has_serial_class or not has_serial_exc:
        print("  ⚠ WARNING: pyserial is broken! Use Anaconda Python kernel.")
except ImportError:
    print("pyserial installed: No - please install: pip install pyserial")

# Find the python_client directory
CLIENT_PATH = os.path.abspath(os.path.join(os.getcwd(), '..', 'platformio_slip', 'python_client'))
if os.path.exists(os.path.join(CLIENT_PATH, 'broccoli_cluster.py')):
    if CLIENT_PATH not in sys.path:
        sys.path.insert(0, CLIENT_PATH)
    print(f'\n✓ Client path: {CLIENT_PATH}')
else:
    print(f'\n✗ broccoli_cluster.py not found at {CLIENT_PATH}')

from broccoli_cluster import BroccoliCluster

# Port configuration
PORT = os.environ.get("BROCCOLI_PORT", "COM25")
print(f'✓ Using port: {PORT}')
print('\n✓ Ready to run tests')

Python: d:\anaconda3\envs\deepLearning\python.exe
pyserial installed: No - please install: pip install pyserial

✗ broccoli_cluster.py not found at c:\Users\Hatsu\Downloads\Assignment1\ESP32-Cluster-cube\platformio_slip\python_client


ModuleNotFoundError: No module named 'broccoli_cluster'

## TEST 1: Connection & Basic Communication

**If pyserial is broken above**, you have two options:
1. **Change kernel** to Anaconda Python (Kernel > Select Another Kernel)
2. **Run from terminal**: `cd platformio_slip/python_client && python test_everything.py`

Otherwise, continue running cells below:

In [ ]:
try:
    cluster = BroccoliCluster(PORT)
    cluster.connect()
    print(f"✓ Serial connection established on {PORT}")
    print("✓ Master node responding")
    connection_ok = True
    time.sleep(0.5)
except Exception as e:
    print(f"✗ Connection failed: {e}")
    print("  → This is expected with MS Store Python (broken pyserial)")
    print("  → To actually run tests:")
    print("     1. Change kernel to Anaconda Python, OR")
    print("     2. Run from terminal: python test_everything.py")
    connection_ok = False
    cluster = None  # Set to None so subsequent cells know connection failed

## TEST 2: Task Definition & Execution

In [ ]:
# Helper function for remaining tests
def skip_if_no_connection():
    if not connection_ok:
        print("⊘ Skipped - no hardware connection")
        return True
    return False

In [ ]:
if not connection_ok:
    print("⊘ Skipped - no hardware connection")
else:
    print("Defining basic tasks...")
    cluster.define_task("add", "lambda a, b: a + b")
    cluster.define_task("multiply", "lambda a, b: a * b")
    cluster.define_task("power", "lambda a, b: a ** b")
    cluster.define_task("square", "lambda x: x * x")
    time.sleep(0.5)

    print("\nExecuting tasks:")
    result = cluster.execute("add", 15, 27)
    print(f"  add(15, 27) = {result}")
    assert result == "42", f"Expected 42, got {result}"

    result = cluster.execute("multiply", 12, 8)
    print(f"  multiply(12, 8) = {result}")
    assert result == "96", f"Expected 96, got {result}"

    result = cluster.execute("power", 2, 10)
    print(f"  power(2, 10) = {result}")
    assert result == "1024", f"Expected 1024, got {result}"

    print("\n✓ All basic task tests passed!")

## TEST 3: Canvas Primitives - GROUP (Parallel Execution)

In [ ]:
if skip_if_no_connection(): pass
else:
    print("Executing group([add(5,3), multiply(4,7), square(9)])...")
    results = cluster.group([
        cluster.sig("add", 5, 3),
        cluster.sig("multiply", 4, 7),
        cluster.sig("square", 9)
    ])
    print(f"Results: {results}")

    expected = [8, 28, 81]
    results_int = [int(x) if isinstance(x, str) else x for x in results]
    assert results_int == expected, f"Expected {expected}, got {results_int}"
    print("✓ GROUP test passed!")

## TEST 4: Canvas Primitives - CHAIN (Pipeline Execution)

In [ ]:
if skip_if_no_connection(): pass
else:
    print("Defining sum_list task...")
    cluster.define_task("sum_list", "lambda lst: sum(lst)")
    time.sleep(0.3)

    print("\nExecuting chain: add(5,3) -> multiply(2) -> square()...")
    result = cluster.chain([
        cluster.sig("add", 5, 3),      # 8
        cluster.sig("multiply", 2),    # 16
        cluster.sig("square")          # 256
    ])
    print(f"Result: {result}")

    result_int = int(result) if isinstance(result, str) else result
    assert result_int == 256, f"Expected 256, got {result_int}"
    print("✓ CHAIN test passed!")

## TEST 5: Canvas Primitives - CHORD (Map-Reduce)
**Note:** This test may fail - CHORD is a known issue (returns None)

In [ ]:
if skip_if_no_connection(): pass
else:
    print("Executing chord: [square(2), square(3), square(4), square(5)] -> sum_list()...")
    try:
        result = cluster.chord(
            header_sigs=[
                cluster.sig("square", 2),   # 4
                cluster.sig("square", 3),   # 9
                cluster.sig("square", 4),   # 16
                cluster.sig("square", 5)    # 25
            ],
            callback_sig=cluster.sig("sum_list")
        )
        print(f"Result: {result}")
        
        if result is None:
            print("⚠ CHORD returned None - Canvas primitive not fully implemented")
        else:
            result_int = int(result) if isinstance(result, str) else result
            assert result_int == 54, f"Expected 54 (4+9+16+25), got {result_int}"
            print("✓ CHORD test passed!")
    except Exception as e:
        print(f"⚠ CHORD test failed (known issue): {e}")

## TEST 6: Dual-Core Execution

In [ ]:
if skip_if_no_connection(): pass
else:
    print("Executing square(50) on Core 0...")
    result = cluster.execute("square", 50, core=0)
    print(f"  Core 0: square(50) = {result}")
    assert result == "2500", f"Expected 2500, got {result}"

    print("\nExecuting square(100) on Core 1...")
    result = cluster.execute("square", 100, core=1)
    print(f"  Core 1: square(100) = {result}")
    assert result == "10000", f"Expected 10000, got {result}"

    print("\n✓ Dual-core execution tests passed!")

## TEST 7: Dual-Core with Canvas GROUP

In [ ]:
if skip_if_no_connection(): pass
else:
    print("Executing group with core assignments...")
    results = cluster.group([
        cluster.sig("square", 10, core=0),
        cluster.sig("square", 20, core=1),
        cluster.sig("square", 30, core=0),
        cluster.sig("square", 40, core=1)
    ])
    print(f"Results: {results}")

    results_int = [int(x) if isinstance(x, str) else x for x in results]
    expected = [100, 400, 900, 1600]
    assert results_int == expected, f"Expected {expected}, got {results_int}"
    print("✓ Dual-core Canvas test passed!")

## TEST 8: Peripheral Initialization

In [ ]:
if skip_if_no_connection(): pass
else:
    print("Initializing peripherals:")
    print("  >> I2C (SDA:21, SCL:22, 100kHz)...")
    cluster.i2c_init(21, 22, 100000)

    print("  >> SPI (SCK:18, MISO:19, MOSI:23, 1MHz)...")
    cluster.spi_init(18, 19, 23, 1000000)

    print("  >> UART (TX:17, RX:16, 115200)...")
    cluster.uart_init(17, 16, 115200)

    print("  >> CAN (TX:5, RX:6, 500kbps)...")
    cluster.can_init(5, 6, 500000)

    print("\n✓ All peripherals initialized!")

## TEST 9: GPIO Control

In [ ]:
if skip_if_no_connection(): pass
else:
    print("Testing GPIO operations:")
    print("  >> Toggling GPIO 46 every second for 4 seconds...")

    for i in range(4):
        state_str = 'HIGH' if i % 2 else 'LOW'
        print(f"    Setting GPIO 46 to {state_str}")
        result = cluster.gpio_write(46, state_str)
        print(f"    GPIO response: {result}")
        time.sleep(1)

    print("\n✓ GPIO control tests passed!")

## TEST 10: ADC Reading

In [ ]:
if skip_if_no_connection(): pass
else:
    print("Reading ADC from GPIO 34...")
    adc_value = cluster.adc_read(34)
    print(f"  ADC value: {adc_value}")
    print("\n✓ ADC reading test passed!")

## TEST 11: System Monitoring

In [ ]:
if skip_if_no_connection(): pass
else:
    print("Retrieving system status...")
    cluster.print_system_status()
    print("\n✓ System monitoring test passed!")

## TEST 12: Dynamic Multiline Code Upload
Upload complex Python code with functions, loops, comments to the worker and execute it remotely!

In [ ]:
if skip_if_no_connection(): pass
else:
    print("Uploading multiline Python code...")

    # Proper multiline Python code with comments, functions, logic
    multiline_code = """# Dynamic GPIO Toggle with proper multiline Python
import machine
import time

def toggle_gpio_advanced(pin_num, cycles=2):
    '''Toggle a GPIO pin multiple times and return status'''
    pin = machine.Pin(pin_num, machine.Pin.OUT)
    
    results = []
    for i in range(cycles):
        # Set LOW
        pin.value(0)
        results.append(f"cycle_{i}_LOW")
        time.sleep_ms(100)
        
        # Set HIGH
        pin.value(1)
        results.append(f"cycle_{i}_HIGH")
        time.sleep_ms(100)
    
    return {
        'status': 'done',
        'pin': pin_num,
        'cycles': cycles,
        'operations': len(results)
    }

# Execute and print result
result = toggle_gpio_advanced(46, cycles=2)
print(result)
"""

    cluster.upload_code("dyntog.py", multiline_code)
    time.sleep(0.5)

    # Define task to import the module (executes on import)
    print("Defining task with import side effect...")
    cluster.define_task("run_dynamic", "__import__('dyntog') or 1")
    time.sleep(0.3)

    print("Executing multiline dynamic code...")
    result = cluster.execute("run_dynamic")
    print(f"  Result: {result}")

    # Test file cleanup
    print("\nTesting file cleanup...")
    cluster.define_task("remove_dynamic", "__import__('os').remove('dyntog.py') or 'removed'")
    time.sleep(0.3)

    print("Removing uploaded file...")
    remove_result = cluster.execute("remove_dynamic")
    print(f"  Cleanup result: {remove_result}")

    if result and "dyntog" in str(result):
        print("\n✓ Dynamic multiline code executed successfully!")
        print("  GPIO 46 was toggled (check hardware)")
    else:
        print(f"  Note: Result was {result}")

## TEST 13: Error Handling

In [ ]:
if skip_if_no_connection(): pass
else:
    print("Testing error scenarios:")

    print("  >> Executing undefined task...")
    try:
        result = cluster.execute("nonexistent_task_xyz", 1, 2)
        print(f"    ✗ Expected error but got: {result}")
    except Exception as e:
        print(f"    ✓ Correctly rejected: {e}")

    print("\n  >> Defining task with syntax error...")
    cluster.define_task("bad_syntax", "this is not valid python code at all")
    time.sleep(0.2)
    try:
        result = cluster.execute("bad_syntax")
        print(f"    ✗ Expected error but got: {result}")
    except Exception as e:
        print(f"    ✓ Correctly handled: {e}")

    print("\n✓ Error handling tests passed!")

## TEST 14: Task Listing & Cleanup

In [ ]:
if skip_if_no_connection(): pass
else:
    print("Listing all defined tasks...")
    tasks = cluster.list_tasks()
    print(f"  Defined tasks: {tasks}")

    if tasks:
        print(f"  Found {len(tasks)} tasks")
    else:
        print("  Note: Task listing may be limited")

    print("\n✓ Task listing test completed")

    # Disconnect
    print("\nDisconnecting from cluster...")
    cluster.disconnect()
    print("✓ Disconnected")

## 🎉 Test Suite Complete!

All tests have been executed. Check the results above:
- ✓ marks indicate successful tests
- ⚠ marks indicate known issues (like CHORD primitive)
- Tests cover: basic tasks, Canvas primitives, dual-core, peripherals, GPIO, ADC, monitoring, dynamic code, and error handling

**Note on Core Assignment:** When you upload dynamic code, you can specify which core runs it:
```python
cluster.execute("run_dynamic", core=0)  # Run on Core 0
cluster.execute("run_dynamic", core=1)  # Run on Core 1
```

**GPIO State:** GPIO pins remain in their last state after code execution until explicitly changed or ESP32 is reset.